# Практика PyTorch

In [1]:
import time
import numpy as np
import torch

print(torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

2.14.0+cu130
cuda NVIDIA GeForce RTX 5060 Ti


# Урок 1. Тензоры и работа с GPU

## 1. Что такое тензор

**Тензор** (`torch.Tensor`) — многомерный массив чисел одного типа. Он похож на `np.ndarray`, но у него есть два важных отличия:

1. тензор может лежать на GPU и считаться там;
2. тензор может участвовать в автоматическом дифференцировании (`autograd`, урок 2).

У тензора есть три основных атрибута:

| Атрибут | Что хранит | Пример |
|---|---|---|
| `x.shape` | размеры по каждой оси | `torch.Size([32, 3, 224, 224])` |
| `x.dtype` | тип элементов | `torch.float32` |
| `x.device` | где лежит тензор | `cpu`, `cuda:0` |

### Создание тензоров

```python
torch.tensor([[1., 2.], [3., 4.]])   # из данных (данные КОПИРУЮТСЯ)
torch.zeros(3, 4)                     # нули
torch.ones(3, 4)                      # единицы
torch.arange(0, 10, 2)                # 0, 2, 4, 6, 8
torch.linspace(0, 1, 5)               # 5 равноотстоящих точек
torch.eye(3)                          # единичная матрица
torch.rand(3, 4)                      # равномерно на [0, 1)
torch.randn(3, 4)                     # нормальное N(0, 1)
torch.zeros_like(x)                   # нули с формой/dtype/device как у x
```

### Связь с NumPy

```python
a = np.arange(6, dtype=np.float32)
t = torch.from_numpy(a)   # t и a ДЕЛЯТ одну память
t[0] = 100                # изменится и a[0]
b = t.numpy()             # обратно, тоже без копирования (только на CPU)
```

`torch.tensor(a)` создаёт копию, а `torch.from_numpy(a)` — нет. Это частый источник ошибок.

## 2. dtype

По умолчанию:
- числа с точкой → `float32`;
- целые числа → `int64`.

Основные типы: `float32` (стандарт для обучения), `float16`/`bfloat16` (ускорение на GPU, урок 7), `float64` (высокая точность), `int64` (индексы, метки классов), `bool` (маски).

```python
x = torch.tensor([1, 2, 3])        # int64
y = x.float()                      # float32
z = x.to(torch.float16)            # явное приведение
```

Арифметика над тензорами разных dtype подчиняется правилам **type promotion**: результат получает более «широкий» тип. Метки классов для `CrossEntropyLoss` должны быть `int64`, а не `float`.

## 3. device и GPU

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x = torch.randn(1000, 1000)          # на CPU
x_gpu = x.to(device)                 # копия на GPU
y_gpu = torch.randn(1000, 1000, device=device)  # сразу создаём на GPU
x_cpu = x_gpu.cpu()                  # обратно на CPU
```

Правила:
- операция возможна только между тензорами **на одном устройстве**; иначе будет `RuntimeError`;
- копирование между CPU и GPU идёт по шине PCIe и стоит дорого, поэтому данные и модель стараются держать на GPU целиком;
- `x.numpy()` работает только для CPU-тензоров (для GPU сначала `.cpu()`);
- `x.item()` достаёт число из тензора с одним элементом (и заставляет GPU дождаться вычислений).

### Асинхронность GPU и измерение времени

Вызовы CUDA **асинхронны**: Python ставит операцию в очередь и идёт дальше, не дожидаясь результата. Если замерить время обычным `time.time()` без синхронизации, получится время постановки в очередь, а не время вычисления. Правильно так:

```python
torch.cuda.synchronize()
t0 = time.perf_counter()
y = x @ x
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
```

Первый запуск на GPU включает инициализацию (прогрев), поэтому перед замером делают 1–2 «разогревочных» вызова.

## 4. Индексация и срезы

Работает так же, как в NumPy:

```python
x = torch.arange(24).reshape(4, 6)
x[0]            # первая строка
x[:, 1]         # второй столбец
x[1:3, ::2]     # срезы с шагом
x[x > 10]       # булева маска -> одномерный тензор
x[[0, 2], [1, 3]]   # целочисленная индексация: элементы (0,1) и (2,3)
x[..., 0]       # Ellipsis: последняя ось = 0, остальные целиком
x[:, None, :]   # добавить ось (None == unsqueeze)
```

Срезы (`x[1:3]`) возвращают **view** (общая память), а индексация списком или маской — **копию**.

## 5. view, reshape, permute: что происходит в памяти

Тензор хранится в памяти как плоский одномерный массив, а форма задаётся метаданными: `shape` и `stride` (на сколько элементов надо шагнуть в памяти, чтобы сдвинуться на 1 по каждой оси).

```python
x = torch.arange(6).reshape(2, 3)
x.stride()          # (3, 1)
x.t().stride()      # (1, 3)  — транспонирование не двигает данные, меняет только stride
x.t().is_contiguous()   # False
```

| Операция | Копирует данные? | Замечание |
|---|---|---|
| `view(...)` | нет | требует совместимых strides, иначе ошибка |
| `reshape(...)` | если нужно | возвращает view, когда возможно, иначе копию |
| `permute`, `transpose`, `t()` | нет | результат обычно не contiguous |
| `contiguous()` | если нужно | делает копию с «правильным» порядком в памяти |
| `squeeze`, `unsqueeze`, `expand` | нет | `expand` не выделяет память |
| `clone()` | да | явная копия |

Практическое правило: после `permute`/`transpose` `view` часто падает, тогда нужен `reshape` или `.contiguous().view(...)`.

Операции с суффиксом `_` (`x.add_(1)`, `x.zero_()`) работают **in-place**. Они изменяют тензор на месте, а значит и все его view. С autograd in-place операции нужно использовать осторожно (урок 2).

## 6. Broadcasting

Когда формы операндов разные, PyTorch выравнивает их **справа налево**. Две размерности совместимы, если они равны, или одна из них равна 1, или её нет. Размерность 1 «растягивается» без копирования памяти.

```
A:     (4, 1, 5)
B:        (3, 5)
A + B: (4, 3, 5)
```

```python
a = torch.randn(4, 1, 5)
b = torch.randn(3, 5)
(a + b).shape    # torch.Size([4, 3, 5])
```

Типичная ловушка:

```python
a = torch.randn(5)
b = torch.randn(5, 1)
(a + b).shape    # (5, 5), а не (5,): вектор и столбец дают матрицу
```

Типичный приём — превратить вычисления в цикле по парам в одну операцию над тензором с добавленной осью:

```python
A = torch.randn(100, 3)   # N точек
B = torch.randn(200, 3)   # M точек
diff = A[:, None, :] - B[None, :, :]   # (100, 200, 3)
dist = diff.pow(2).sum(-1).sqrt()      # (100, 200)
```

## 7. Редукции и `einsum`

Редукции (`sum`, `mean`, `max`, `min`, `argmax`, `std`, `norm`) принимают `dim` (ось) и `keepdim`:

```python
x = torch.randn(4, 5)
x.sum(dim=1).shape                 # (4,)
x.sum(dim=1, keepdim=True).shape   # (4, 1)  — удобно для последующего broadcasting
```

`torch.einsum` описывает операцию над осями строкой:

```python
torch.einsum("ij,jk->ik", A, B)         # матричное умножение
torch.einsum("bij,bjk->bik", A, B)      # батчевое матричное умножение
torch.einsum("bi,bi->b", u, v)          # скалярные произведения по батчу
```

Индексы, которых нет справа от `->`, суммируются. Для матричных умножений есть `@` (`torch.matmul`) и `torch.bmm`; они обычно быстрее и читабельнее на простых случаях.

## 8. Числовая устойчивость

Формулы на бумаге и в коде ведут себя по-разному. Пример — **softmax**:

$$\mathrm{softmax}(x)_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

Для больших $x_i$ значение $e^{x_i}$ переполняется до `inf`, и получается `nan`. Решение — вычесть максимум (результат не изменится, так как числитель и знаменатель домножаются на одну и ту же константу):

$$\mathrm{softmax}(x)_i = \frac{e^{x_i - m}}{\sum_j e^{x_j - m}}, \quad m = \max_j x_j$$

Похожая проблема при вычислении расстояний через формулу $\|a-b\|^2 = \|a\|^2 + \|b\|^2 - 2\,a\cdot b$: из-за ошибок округления результат может получиться чуть меньше нуля, и `sqrt` даст `nan`. Перед корнем значения обрезают снизу нулём (`clamp(min=0)`).

## Задачи урока 1

### Задача 1.1. Попарные расстояния тремя способами

Даны `A` формы `(N, D)` и `B` формы `(M, D)`. Нужно получить матрицу `(N, M)` евклидовых расстояний.

Реализуйте три функции:
1. `dist_loops(A, B)`: два вложенных цикла по строкам;
2. `dist_broadcast(A, B)`: через broadcasting с добавлением осей;
3. `dist_matmul(A, B)`: через формулу $\|a\|^2 + \|b\|^2 - 2ab^\top$ и матричное умножение. Защититесь от отрицательных значений под корнем.

Проверьте все три против `torch.cdist(A, B)`. Затем для `N = M = 2000`, `D = 128` на CPU и на GPU замерьте время (с `synchronize` и прогревом) и объясните разницу между вариантами 2 и 3 (подсказка: посмотрите, какой промежуточный тензор создаёт каждый из них).

### Задача 1.2. Устойчивый softmax

Реализуйте `softmax(x, dim)` вручную с вычитанием максимума. Покажите на тензоре с большими значениями (например, `x = torch.tensor([[1000., 1001., 1002.]])`), что наивная версия `exp(x) / exp(x).sum(...)` выдаёт `nan`, а ваша совпадает с `torch.softmax`. Проверьте, что работает для произвольной оси `dim`, в том числе отрицательной.

### Задача 1.3. kNN в батче без циклов

Дан тензор `X` формы `(B, N, D)`: `B` облаков по `N` точек. Для каждой точки найдите `K` ближайших соседей **внутри её облака** (саму точку не считать соседом) и верните:
- индексы соседей `idx` формы `(B, N, K)`;
- координаты соседей `nbrs` формы `(B, N, K, D)`.

Циклов по батчу и по точкам быть не должно. Подсказки: батчевая матрица расстояний `(B, N, N)`, `torch.topk(..., largest=False)`, для сбора координат — продвинутая индексация или `torch.gather`. Проверьте результат на небольшом примере наивной реализацией с циклами.

### Задача 1.4. Когда GPU выигрывает

Для квадратных матриц размеров `n ∈ {64, 256, 1024, 4096}` замерьте время `A @ B` на CPU и на GPU (по 10 повторов после прогрева, правильно синхронизируя GPU). Составьте таблицу и ответьте письменно:
- при каком `n` GPU становится быстрее и почему на малых размерах он может проигрывать;
- что изменится в замерах, если включить в них время копирования `A` и `B` на GPU.

### Задача 1.5. Что изменится в памяти (предсказать, потом проверить)

Для каждого фрагмента сначала запишите ответ в комментарии, потом запустите и сравните.

```python
# (а)
x = torch.arange(12).reshape(3, 4)
y = x.t()
y[0, 0] = 100
# Чему равно x[0, 0]? Почему?

# (б)
x = torch.arange(12).reshape(3, 4)
y = x.t().view(12)
# Что произойдёт? Как исправить двумя способами и чем они различаются?

# (в)
x = torch.arange(12).reshape(3, 4)
y = x[x > 5]
y[0] = -1
# Изменится ли x? Почему?

# (г)
a = torch.randn(5)
b = torch.randn(5, 1)
c = a * b
# Какая форма у c? Какую ошибку чаще всего допускают с таким кодом?
```